# 03 — Features

This is the single, growing feature-engineering notebook (informed by `02_eda.ipynb`, which runs first). Every new feature gets added here as a new section, reading `data/interim/` inputs from `01_preprocessing.ipynb` and writing one cumulative feature file other notebooks build on.

Built so far: floor area (as of sale date), inflation adjustment, geocoding, `relative_size`, house type, coastal distance.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.config import DATA_RAW, DATA_INTERIM, LOCAL_AUTHORITIES
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

joined = pd.read_parquet(DATA_INTERIM / "ppd_epc_joined.parquet")
epc = pd.read_parquet(DATA_INTERIM / "epc_clean.parquet")
print(f"{len(joined):,} joined transactions | {len(epc):,} EPC certificates")
print(joined["district"].value_counts().to_string())


## Part A0 — attach each sale's own floor area (as of the sale date)

Notebook 03's join found *which* property each sale matches, but didn't
carry the property's floor area, rooms etc. through — that was done
separately, just for the one-off baseline fit, in notebook 03b. Doing it
properly here so every later feature can build on it.

Same rule as before: a property can have several EPC certificates over
time, so for each sale we take the certificate **current as of that sale
date** (plan Part 5, leakage control #1) — never one issued after, since
that could reflect a renovation the buyer made *after* buying.


In [2]:
epc_sorted = epc.dropna(subset=["UPRN"]).sort_values("inspection_date")
epc_groups = {uprn: grp for uprn, grp in epc_sorted.groupby("UPRN")}

def certificate_as_of(uprn, sale_date):
    candidates = epc_groups.get(uprn)
    if candidates is None:
        return None
    valid = candidates[candidates["inspection_date"] <= sale_date]
    return valid.iloc[-1] if not valid.empty else candidates.iloc[0]

attach_cols = ["total_floor_area", "number_habitable_rooms", "number_heated_rooms",
               "property_type", "built_form", "construction_age_band", "tenure"]

rows = []
for row in joined.dropna(subset=["uprn_final"]).itertuples(index=False):
    cert = certificate_as_of(row.uprn_final, row.date_of_transfer)
    d = {"transaction_id": row.transaction_id}
    if cert is not None:
        d.update({c: cert[c] for c in attach_cols})
    rows.append(d)

epc_at_sale = pd.DataFrame(rows)
joined = joined.merge(epc_at_sale, on="transaction_id", how="left")
print(f"Floor area attached for {joined['total_floor_area'].notna().mean():.1%} of joined sales")


Floor area attached for 86.3% of joined sales


## Part A — inflation adjustment

**In plain terms:** the UK HPI is an official monthly score for how expensive
housing is in an area, published by HM Land Registry/ONS. It's not itself a
price — it's an index number (think of it like a thermometer reading, not a
temperature in a specific room). If the index was 50 when a house sold and
it's 110 today, that area's prices have roughly doubled since then, so we
scale that historic sale price up by 110/50 to say what an equivalent house
would sell for *today*. This puts every sale, regardless of year, on the
same footing before the model ever sees it.


In [ ]:
hpi = pd.read_csv(
    DATA_RAW / "hpi" / "uk-hpi-full-2026-05.csv",
    usecols=["Date", "RegionName", "Index"],
)
hpi["Date"] = pd.to_datetime(hpi["Date"], format="%d/%m/%Y")

# One index series per authority. Applying Sefton's price growth to
# Liverpool sales would inject error rather than remove it - these markets
# have not moved at the same rate.
hpi = hpi[hpi["RegionName"].isin(LOCAL_AUTHORITIES.values())].copy()
region_to_district = {v: k for k, v in LOCAL_AUTHORITIES.items()}
hpi["district"] = hpi["RegionName"].map(region_to_district)
hpi = hpi.sort_values("Date").reset_index(drop=True)

latest_date = hpi["Date"].max()
latest = hpi[hpi["Date"] == latest_date].set_index("district")["Index"]
print(f"Adjusting every sale to {latest_date.date()} prices, per authority:")
print(latest.to_string())


In [ ]:
# Match each sale to its OWN authority's index for that month.
# merge_asof needs both sides sorted by the join key; `by=` keeps the
# authorities from bleeding into each other.
joined = joined.sort_values("date_of_transfer")
hpi_sorted = hpi.sort_values("Date")

joined = pd.merge_asof(
    joined, hpi_sorted[["Date", "Index", "district"]],
    left_on="date_of_transfer", right_on="Date",
    by="district", direction="backward",
)

joined["latest_index"] = joined["district"].map(latest)
joined["price_adjusted"] = joined["price"] * (joined["latest_index"] / joined["Index"])

unmatched = joined["Index"].isna().sum()
print(f"{unmatched:,} sales couldn't be matched to an HPI month (should be ~0)")
joined[["district", "date_of_transfer", "price", "Index", "price_adjusted"]].head()


## Check: does the adjustment actually flatten the price-growth trend?

If it worked, median *adjusted* price by year should be roughly flat,
unlike the raw median which climbed from £147,500 to £240,000.


In [ ]:
joined["year"] = joined["date_of_transfer"].dt.year
by_year = joined.groupby("year")[["price", "price_adjusted"]].median()
print(by_year)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
by_year["price"].plot(ax=axes[0], marker="o", label="raw price")
by_year["price_adjusted"].plot(ax=axes[0], marker="o", label="HPI-adjusted")
axes[0].set_ylabel("Median price (£)")
axes[0].set_title("All authorities combined")
axes[0].legend()

for district, grp in joined.groupby("district"):
    grp.groupby("year")["price_adjusted"].median().plot(ax=axes[1], marker=".", label=district)
axes[1].set_ylabel("Median adjusted price (£)")
axes[1].set_title("Adjusted price by authority (should each be ~flat)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## Part B — geocoding

**In plain terms:** postcodes.io is a free, no-signup lookup that turns a UK
postcode into a latitude/longitude plus some official geography codes (which
statistical small-area it falls in, its deprivation rank, etc). It has a
bulk endpoint that accepts up to 100 postcodes per request, so we batch
through every unique postcode rather than looking each one up individually.

We geocode the postcodes from the **full EPC certificate set** (116,070
rows), not just the 61,344 that matched a sale — `relative_size` needs to
know about nearby houses whether or not they've ever sold.


In [6]:
unique_postcodes = epc["POSTCODE"].dropna().unique().tolist()
print(f"{len(unique_postcodes):,} unique postcodes to geocode")


5,959 unique postcodes to geocode


In [ ]:
def geocode_batch(postcodes: list[str]) -> pd.DataFrame:
    resp = requests.post(
        "https://api.postcodes.io/postcodes",
        json={"postcodes": postcodes},
        timeout=30,
    )
    resp.raise_for_status()
    body = resp.json()
    rows = []
    for item in body["result"]:
        pc = item["query"]
        r = item.get("result")
        if r is None:
            continue
        rows.append({
            "postcode": pc,
            "lat": r["latitude"],
            "lon": r["longitude"],
            "lsoa": r["lsoa"],
            "msoa": r["msoa"],
            "admin_ward": r["admin_ward"],
        })
    return pd.DataFrame(rows)

geo_cache = DATA_INTERIM / "postcode_geocodes.parquet"
if geo_cache.exists():
    geocodes = pd.read_parquet(geo_cache)
    print(f"Loaded {len(geocodes):,} cached geocodes")
else:
    batches = [unique_postcodes[i:i+100] for i in range(0, len(unique_postcodes), 100)]
    results = []
    for i, batch in enumerate(batches):
        results.append(geocode_batch(batch))
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(batches)} batches done")
    geocodes = pd.concat(results, ignore_index=True)
    geocodes.to_parquet(geo_cache, index=False)
    print(f"Geocoded {len(geocodes):,} / {len(unique_postcodes):,} postcodes ({len(geocodes)/len(unique_postcodes):.1%})")


## Attach coordinates and save

Both the sale-matched dataset and the full EPC set get coordinates, since
Part A's output feeds the model directly and Part B's fuller set feeds
`relative_size`.


In [8]:
joined_geo = joined.merge(geocodes, left_on="postcode", right_on="postcode", how="left")
epc_geo = epc.merge(geocodes, left_on="POSTCODE", right_on="postcode", how="left")

print(f"Sales with coordinates: {joined_geo['lat'].notna().mean():.1%}")
print(f"EPC certificates with coordinates: {epc_geo['lat'].notna().mean():.1%}")


Sales with coordinates: 99.9%
EPC certificates with coordinates: 99.8%


## Part C — `relative_size`: how big is this house compared to its neighbours?

This is the main idea we're testing (plan Part 2). The original buyer's
report found that on streets where every house is roughly the same size,
floor area alone stops predicting price at all (R² collapsed from 0.88 on
varied streets to 0.076 on a uniform one) — because there's no variation
left for it to explain. `relative_size` reframes the same number: instead
of "this house is 120 m²," it says "this house is 35% bigger than the
average of the 15 nearest properties" — which still carries information
even when every house nearby is a similar size, and means the same 120 m²
is read differently on a street of 80 m² houses vs a street of 150 m² ones.

**How "nearest" is found:** convert latitude/longitude to plain
east/north metres on the **British National Grid** (the coordinate system
British maps and other UK features in this project use), then use a
**KD-tree** — a standard data structure for "find the k closest points"
that's far faster than checking every pair of houses by hand — to pull the
15 nearest properties to each sale, by straight-line distance.

**The neighbour pool is the full EPC set** (116,070 certificates, not just
sold ones), deduplicated to each property's most recent floor area, so a
house that has never sold can still act as a "neighbour" for comparison.


In [9]:
from scipy.spatial import cKDTree
import pyproj

K = 15  # matches sqft_living15 in the reference project

neighbour_pool = epc_geo.dropna(subset=["UPRN", "lat", "lon", "total_floor_area"]).copy()
neighbour_pool = neighbour_pool.sort_values("inspection_date").drop_duplicates(subset=["UPRN"], keep="last")
print(f"{len(neighbour_pool):,} unique properties in the neighbour pool")

# EPSG:4326 = plain lat/lon (what we have); EPSG:27700 = British National Grid,
# metres on a flat plane (what we need for real distances - plan Part 2 flags
# raw lat/lon degrees as "the classic error", same reasoning applies here).
transformer = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)
easting, northing = transformer.transform(neighbour_pool["lon"].values, neighbour_pool["lat"].values)
neighbour_pool = neighbour_pool.assign(easting=easting, northing=northing)

tree = cKDTree(neighbour_pool[["easting", "northing"]].to_numpy())
pool_uprns = neighbour_pool["UPRN"].to_numpy()
pool_areas = neighbour_pool["total_floor_area"].to_numpy()
uprn_to_pool_idx = {u: i for i, u in enumerate(pool_uprns)}


85,696 unique properties in the neighbour pool


In [10]:
def neighbourhood_mean_area(lat, lon, own_uprn, k=K):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan
    e, n = transformer.transform(lon, lat)
    # ask for k+1 in case the property itself is in the pool, then drop it
    dists, idxs = tree.query([e, n], k=k + 1)
    own_idx = uprn_to_pool_idx.get(own_uprn)
    idxs = [i for i in idxs if i != own_idx][:k]
    if not idxs:
        return np.nan
    return pool_areas[idxs].mean()

joined_geo["neighbourhood_mean_area"] = [
    neighbourhood_mean_area(row.lat, row.lon, row.uprn_final)
    for row in joined_geo.itertuples(index=False)
]
joined_geo["relative_size"] = joined_geo["total_floor_area"] / joined_geo["neighbourhood_mean_area"]

print(f"relative_size computed for {joined_geo['relative_size'].notna().mean():.1%} of sales")
print(joined_geo["relative_size"].describe())


relative_size computed for 86.3% of sales
count    52928.000000
mean         1.045497
std          0.358106
min          0.129487
25%          0.868790
50%          0.988121
75%          1.140055
max         10.360825
Name: relative_size, dtype: float64


## Does it correlate with price better than raw floor area?

The real test. If the idea works, `relative_size` should hold up (or do
better) specifically on the streets where raw floor area struggles.


In [11]:
have_both = joined_geo.dropna(subset=["relative_size", "total_floor_area", "price_adjusted"])
log_price = np.log(have_both["price_adjusted"])
print(f"log(price) vs log(floor_area):    corr = {np.corrcoef(np.log(have_both['total_floor_area']), log_price)[0,1]:+.3f}")
print(f"log(price) vs log(relative_size): corr = {np.corrcoef(np.log(have_both['relative_size']), log_price)[0,1]:+.3f}")


log(price) vs log(floor_area):    corr = +0.695
log(price) vs log(relative_size): corr = +0.253


## Save the full feature set


In [ ]:
joined_geo.to_parquet(DATA_INTERIM / "ppd_epc_joined_features.parquet", index=False)
epc_geo.to_parquet(DATA_INTERIM / "epc_sefton_geocoded.parquet", index=False)
print("Saved.")


---
## House type and coastal distance

In [ ]:
import sys
sys.path.insert(0, "..")
from src.config import DATA_RAW, DATA_INTERIM
import pandas as pd
import numpy as np
import geopandas as gpd
import statsmodels.api as sm

df = pd.read_parquet(DATA_INTERIM / "ppd_epc_joined_features.parquet")
print(f"{len(df):,} transactions with features so far")


## Part A — house type

`built_form` is a category (Detached, Semi-Detached, Mid-Terrace, ...), not
a number, so a straight-line model needs it turned into a set of yes/no
columns — e.g. `is_detached` (1 if detached, 0 otherwise) — one per
category. This is called **one-hot encoding**. We drop one category
(terraces, the most common) as the "baseline" that the others are compared
against, which is standard practice and avoids redundant columns.


In [14]:
# Collapse rare sub-categories (Enclosed Mid/End-Terrace) into their plain
# counterparts - EDA showed them behaving similarly and some have very few
# examples, which makes their own dummy coefficient unstable.
built_form_map = {
    "Enclosed Mid-Terrace": "Mid-Terrace",
    "Enclosed End-Terrace": "End-Terrace",
}
df["built_form_clean"] = df["built_form"].replace(built_form_map)

dummies = pd.get_dummies(df["built_form_clean"], prefix="type", drop_first=True, dtype=float)
print("Categories (one dropped as baseline):", dummies.columns.tolist())
df = pd.concat([df, dummies], axis=1)


Categories (one dropped as baseline): ['type_End-Terrace', 'type_Mid-Terrace', 'type_Not Recorded', 'type_Semi-Detached']


## Part B — coastal distance

**Why British National Grid, not raw lat/lon.** A degree of longitude
covers a different real-world distance than a degree of latitude (more so
the further from the equator), so measuring "distance" directly in lat/lon
numbers gives a distorted answer. EPSG:27700 (British National Grid) is a
flat-earth projection designed for exactly this — coordinates in it are
plain metres on a grid, the same system British maps use.

**Why beach polygons, not the general coastline.** Sefton's shoreline runs
from Crosby beach into Seaforth container terminal further south. Measuring
to the nearest bit of *any* coastline would score a house backing onto a
working port the same as one facing the beach. Beach polygons are tagged
in OpenStreetMap specifically as `natural=beach` — nobody tags a container
terminal that way, so this measure only ever finds genuine open beach.


In [15]:
osm_path = DATA_RAW / "osm" / "merseyside-latest.osm.pbf"

beaches = gpd.read_file(osm_path, layer="multipolygons", where="natural = 'beach'")
print(f"{len(beaches)} beach polygons found")

marine_lakes = gpd.read_file(osm_path, layer="multipolygons", where="natural = 'water'")
marine_lakes = marine_lakes[marine_lakes["name"].str.contains("Marine Lake", case=False, na=False)]
print(f"{len(marine_lakes)} Marine Lake polygons found")


72 beach polygons found


4 Marine Lake polygons found


In [16]:
BNG = "EPSG:27700"
beaches_bng = beaches.to_crs(BNG)
lakes_bng = marine_lakes.to_crs(BNG)

# One combined shape per feature type - much faster to measure distance to
# a single unified shape than to loop over 72 separate beach polygons.
beach_shape = beaches_bng.union_all()
lake_shape = lakes_bng.union_all()

properties = df.dropna(subset=["lat", "lon"]).copy()
points = gpd.GeoSeries(
    gpd.points_from_xy(properties["lon"], properties["lat"]), crs="EPSG:4326"
).to_crs(BNG)

properties["dist_to_beach_m"] = points.distance(beach_shape).values
properties["dist_to_marine_lake_m"] = points.distance(lake_shape).values

print(properties[["dist_to_beach_m", "dist_to_marine_lake_m"]].describe())


       dist_to_beach_m  dist_to_marine_lake_m
count     61310.000000           61310.000000
mean       3081.014261            3903.019049
std        2104.657078            2761.469396
min          42.709885              44.814826
25%        1651.344208            1832.563479
50%        2553.298208            2931.566211
75%        3739.382016            5918.191789
max       10508.958636           10487.768087


## Sanity checks

1. No property should show as essentially on top of the beach *and* be in
   Bootle/Seaforth (postcodes L20/L21, where the docks are) — that would
   mean the docks were accidentally being scored as beach.
2. Beach and Marine Lake distance were flagged in the plan as likely
   redundant (~200 m apart in reality) — check the correlation between them
   before deciding whether to keep both.


In [17]:
dock_area = properties[properties["postcode"].str.startswith(("L20", "L21"))]
print(f"Closest a Bootle/Seaforth (L20/L21) property gets to 'beach': "
      f"{dock_area['dist_to_beach_m'].min():.0f} m "
      f"(should be a real distance, not ~0)")

corr = properties[["dist_to_beach_m", "dist_to_marine_lake_m"]].corr().iloc[0, 1]
print(f"\nCorrelation between beach distance and Marine Lake distance: {corr:+.3f}")
print("Keeping both if clearly separate, dropping one if this is above ~0.9 as expected.")


Closest a Bootle/Seaforth (L20/L21) property gets to 'beach': 791 m (should be a real distance, not ~0)

Correlation between beach distance and Marine Lake distance: +0.557
Keeping both if clearly separate, dropping one if this is above ~0.9 as expected.


## Does coastal distance actually correlate with price?


In [18]:
have_price = properties.dropna(subset=["price_adjusted"])
log_price = np.log(have_price["price_adjusted"])
corr_beach = np.corrcoef(have_price["dist_to_beach_m"], log_price)[0, 1]
print(f"log(price) vs distance to beach: corr = {corr_beach:+.3f}  "
      f"(negative expected - further from the beach, cheaper)")


log(price) vs distance to beach: corr = -0.064  (negative expected - further from the beach, cheaper)


## Save the full feature set

In [ ]:
properties.to_parquet(DATA_INTERIM / "features.parquet", index=False)
print(f"Saved {len(properties):,} rows, {properties.shape[1]} columns to features.parquet")